In [8]:
import pandas as pd
import sqlite3
from pathlib import Path

projet = Path(r"C:\Projets\permis-quebec")
fichier = list((projet / "data" / "raw").glob("*.csv"))[0]

df = pd.read_csv(fichier, low_memory=False)
df.shape

(71612, 10)

In [9]:
base = projet / "data" / "clean" / "permis.db"
connexion = sqlite3.connect(base)

df.to_sql("permis_raw", connexion, if_exists="replace", index=False)
print("Base créée :", base)

Base créée : C:\Projets\permis-quebec\data\clean\permis.db


In [3]:
pd.read_sql("SELECT COUNT(*) AS nb_permis FROM permis_raw", connexion)

,nb_permis
0,71612


In [4]:
script = (projet / "sql" / "01_create_tables.sql").read_text(encoding="utf-8")
connexion.executescript(script)
connexion.commit()
print("Table permis créée")

Table permis créée


In [5]:
pd.read_sql("""
    SELECT arrondissement, COUNT(*) AS nb_permis
    FROM permis
    GROUP BY arrondissement
    ORDER BY nb_permis DESC
""", connexion)

,arrondissement,nb_permis
0,Sainte-Foy–Sillery–Cap-Rouge,16421
1,La Haute-Saint-Charles,13303
2,La Cité-Limoilou,13150
3,Beauport,10459
4,Charlesbourg,8640
5,Les Rivières,7311
6,Hors arrondissements,25


In [6]:
def executer_sql(nom_fichier):
    texte = (projet / "sql" / nom_fichier).read_text(encoding="utf-8")
    for requete in texte.split(";"):
        if requete.strip():
            display(pd.read_sql(requete, connexion))

In [7]:
executer_sql("02_controles_qualite.sql")

,nb_lignes
0,69309


,nb_numeros_en_double
0,0


,dates_manquantes,dates_futures,premiere_date,derniere_date
0,0,0,2020-01-03,2026-11-13


,sans_numero,sans_type,sans_raison,sans_lot
0,3,8,6,177


In [10]:
indicateurs = {
    "permis_par_mois": """
        SELECT mois, COUNT(*) AS nb_permis
        FROM permis
        GROUP BY mois
        ORDER BY mois
    """,
    "permis_par_type": """
        SELECT COALESCE(type_permis, 'Type non précisé') AS type_permis,
               COUNT(*) AS nb_permis,
               ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM permis), 1) AS part_pct
        FROM permis
        GROUP BY type_permis
        ORDER BY nb_permis DESC
    """,
    "permis_par_domaine": """
        SELECT domaine, COUNT(*) AS nb_permis
        FROM permis
        GROUP BY domaine
        ORDER BY nb_permis DESC
    """,
    "permis_par_arrondissement_annee": """
        SELECT arrondissement, annee, COUNT(*) AS nb_permis
        FROM permis
        GROUP BY arrondissement, annee
        ORDER BY arrondissement, annee
    """,
    "variation_mensuelle": """
        SELECT mois, nb_permis,
               LAG(nb_permis) OVER (ORDER BY mois) AS nb_mois_precedent,
               ROUND(100.0 * (nb_permis - LAG(nb_permis) OVER (ORDER BY mois))
                     / LAG(nb_permis) OVER (ORDER BY mois), 1) AS variation_pct
        FROM (SELECT mois, COUNT(*) AS nb_permis FROM permis GROUP BY mois)
        ORDER BY mois
    """,
}

for nom, requete in indicateurs.items():
    resultat = pd.read_sql(requete, connexion)
    resultat.to_csv(projet / "data" / "clean" / f"{nom}.csv", index=False, encoding="utf-8-sig")
    print(nom, ":", len(resultat), "lignes")

permis_par_mois : 83 lignes
permis_par_type : 5 lignes
permis_par_domaine : 10 lignes
permis_par_arrondissement_annee : 49 lignes
variation_mensuelle : 83 lignes
